# 03 — Gold | Players by Country (Dominicans in MLB in focus)

**Project:** MLB Performance Intelligence Platform

The Dominican Republic is the largest source of foreign-born MLB players. This notebook turns
the `birth_country` attribute into a dedicated analysis (KPI catalog DR-001 … DR-006).

| Object | Type | Question it answers |
|---|---|---|
| `gold.country_season_summary` | table | How many players and how much production (HR, hits, strikeouts, wins) comes from each country, and what share of the league is it? |
| `gold.dominican_batting_season` | view | Ranking of Dominican hitters by season (OPS, HR, RBI) with their league rank |
| `gold.dominican_pitching_season` | view | Ranking of Dominican pitchers by season (ERA, SO, saves) |
| `gold.latest_game_day_lines` | view | How did every player (any country) do on the most recent game day? |
| `gold.dominican_latest_games` | view | Same, filtered to Dominican-born players |

In [ ]:
import sys, importlib
_nb = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
sys.path.insert(0, "/Workspace" + _nb.rsplit("/", 2)[0] + "/shared")
import config as C
import pipeline as P
for _m in (C, P):
    importlib.reload(_m)

RUN_ID = P.new_run_id()
T = lambda s, n: C.table(s, n)

In [ ]:
country = spark.sql(f"""
WITH bat AS (
  SELECT season, season_type, COALESCE(birth_country, 'Unknown') AS birth_country,
         COUNT(*) AS hitters, SUM(pa) AS pa, SUM(h) AS h, SUM(hr) AS hr, SUM(rbi) AS rbi, SUM(sb) AS sb
  FROM {T('gold', 'player_season_batting')} WHERE pa > 0 GROUP BY ALL
),
pit AS (
  SELECT season, season_type, COALESCE(birth_country, 'Unknown') AS birth_country,
         COUNT(*) AS pitchers, SUM(outs) AS outs, SUM(w) AS w, SUM(sv) AS sv, SUM(so) AS so, SUM(er) AS er
  FROM {T('gold', 'player_season_pitching')} GROUP BY ALL
),
players AS (  -- distinct players (two-way players counted once)
  SELECT season, season_type, COALESCE(birth_country, 'Unknown') AS birth_country, COUNT(DISTINCT player_id) AS players
  FROM (SELECT season, season_type, birth_country, player_id FROM {T('gold', 'player_season_batting')} WHERE pa > 0
        UNION SELECT season, season_type, birth_country, player_id FROM {T('gold', 'player_season_pitching')})
  GROUP BY ALL
),
j AS (
  SELECT p.season, p.season_type, p.birth_country, p.players,
         COALESCE(b.hitters, 0) AS hitters, COALESCE(pt.pitchers, 0) AS pitchers,
         COALESCE(b.h, 0) AS h, COALESCE(b.hr, 0) AS hr, COALESCE(b.rbi, 0) AS rbi, COALESCE(b.sb, 0) AS sb,
         COALESCE(pt.w, 0) AS pitcher_wins, COALESCE(pt.sv, 0) AS saves, COALESCE(pt.so, 0) AS pitcher_strikeouts,
         ROUND(27 * pt.er / NULLIF(pt.outs, 0), 2) AS era
  FROM players p
  LEFT JOIN bat b USING (season, season_type, birth_country)
  LEFT JOIN pit pt USING (season, season_type, birth_country)
)
SELECT *,
  ROUND(100 * players / SUM(players) OVER (PARTITION BY season, season_type), 1) AS pct_of_players,
  ROUND(100 * hr / NULLIF(SUM(hr) OVER (PARTITION BY season, season_type), 0), 1) AS pct_of_home_runs,
  ROUND(100 * pitcher_strikeouts / NULLIF(SUM(pitcher_strikeouts) OVER (PARTITION BY season, season_type), 0), 1) AS pct_of_strikeouts,
  RANK() OVER (PARTITION BY season, season_type ORDER BY players DESC) AS country_rank_by_players
FROM j
""")

with P.log_step(spark, RUN_ID, "gold", "country_season_summary") as step:
    step.records = P.replace_table(spark, country, T("gold", "country_season_summary"),
        "Gold: players and production by birth country per season and season type, with share of league totals")

In [ ]:
spark.sql(f"""
CREATE OR REPLACE VIEW {T('gold', 'dominican_batting_season')}
COMMENT 'Gold: Dominican-born hitters per season with rank among Dominicans and league ranks'
AS SELECT *,
  RANK() OVER (PARTITION BY season, season_type ORDER BY ops DESC NULLS LAST) AS dominican_ops_rank,
  RANK() OVER (PARTITION BY season, season_type ORDER BY hr DESC) AS dominican_hr_rank
FROM {T('gold', 'player_season_batting')}
WHERE is_dominican AND pa > 0
""")

spark.sql(f"""
CREATE OR REPLACE VIEW {T('gold', 'dominican_pitching_season')}
COMMENT 'Gold: Dominican-born pitchers per season with rank among Dominicans and league ranks'
AS SELECT *,
  RANK() OVER (PARTITION BY season, season_type ORDER BY so DESC) AS dominican_so_rank,
  RANK() OVER (PARTITION BY season, season_type ORDER BY sv DESC) AS dominican_sv_rank
FROM {T('gold', 'player_season_pitching')}
WHERE is_dominican
""")

spark.sql(f"""
CREATE OR REPLACE VIEW {T('gold', 'latest_game_day_lines')}
COMMENT 'Gold: batting and pitching line of every player on the most recent game day, with birth country'
AS WITH last_day AS (SELECT MAX(official_date) AS d FROM {T('silver', 'fact_batting_game')})
SELECT 'Batting' AS line_type, b.official_date, b.game_pk, p.player_id, p.player_name, p.birth_country, p.is_dominican,
       t.team_abbreviation, o.team_abbreviation AS opponent, b.position,
       CONCAT(b.h, '-', b.ab, CASE WHEN b.hr > 0 THEN CONCAT(', ', b.hr, ' HR') ELSE '' END,
              CASE WHEN b.rbi > 0 THEN CONCAT(', ', b.rbi, ' RBI') ELSE '' END,
              CASE WHEN b.bb > 0 THEN CONCAT(', ', b.bb, ' BB') ELSE '' END) AS line,
       b.h AS hits, b.hr AS home_runs, b.rbi, NULL AS innings, NULL AS strikeouts, NULL AS earned_runs, NULL AS decision
FROM {T('silver', 'fact_batting_game')} b
JOIN last_day ON b.official_date = last_day.d
JOIN {T('silver', 'dim_player')} p ON p.player_id = b.player_id
LEFT JOIN {T('silver', 'dim_team')} t ON t.team_id = b.team_id
LEFT JOIN {T('silver', 'dim_team')} o ON o.team_id = b.opponent_id
UNION ALL
SELECT 'Pitching', pg.official_date, pg.game_pk, p.player_id, p.player_name, p.birth_country, p.is_dominican,
       t.team_abbreviation, o.team_abbreviation, 'P',
       CONCAT(pg.innings_pitched_display, ' IP, ', pg.er, ' ER, ', pg.so, ' SO', COALESCE(CONCAT(' (', pg.decision, ')'), '')),
       NULL, NULL, NULL, pg.innings_pitched_display, pg.so, pg.er, pg.decision
FROM {T('silver', 'fact_pitching_game')} pg
JOIN last_day ON pg.official_date = last_day.d
JOIN {T('silver', 'dim_player')} p ON p.player_id = pg.player_id
LEFT JOIN {T('silver', 'dim_team')} t ON t.team_id = pg.team_id
LEFT JOIN {T('silver', 'dim_team')} o ON o.team_id = pg.opponent_id
""")

spark.sql(f"""
CREATE OR REPLACE VIEW {T('gold', 'dominican_latest_games')}
COMMENT 'Gold: batting and pitching lines of every Dominican player on the most recent game day'
AS SELECT * FROM {T('gold', 'latest_game_day_lines')} WHERE is_dominican
""")

In [ ]:
display(spark.sql(f"""
    SELECT season, season_type, birth_country, players, pct_of_players, hr, pct_of_home_runs, pitcher_strikeouts
    FROM {T('gold', 'country_season_summary')}
    WHERE season_type = 'Regular Season' AND country_rank_by_players <= 5 ORDER BY season DESC, players DESC"""))
display(spark.table(T("gold", "dominican_latest_games")))